## EDA on label from User

QUESTION 1: Find all possible unique values
- Category
- Subcategory
- Root_code
- Sub_code

In [ ]:
import pandas as pd

In [ ]:

df = pd.read_excel('label.xlsx')

print("--- 1. UNIQUE VALUES IN DATASET ---")
print(f"Categories ({df['Category'].nunique()}): {df['Category'].dropna().unique()}")
print(f"Subcategories ({df['Subcategory'].nunique()}): {df['Subcategory'].dropna().unique()}")
print(f"Root Codes ({df['root_code'].nunique()}): {df['root_code'].dropna().unique()}")
print(f"Sub Codes ({df['sub_code'].nunique()}): {df['sub_code'].dropna().unique()}\n")

QUESTION 2: Find mappings and human errors

In [ ]:
print("--- 2. MAPPING RULES & ANOMALY DETECTION ---")

# Group by the human text labels and see what codes they mapped to, counting occurrences
mapping_df = df.groupby(
    ['Category', 'Subcategory', 'root_code', 'sub_code'], 
    dropna=False
).size().reset_index(name='Count')

# Sort so that for each Category/Subcategory, the most frequent mapping is at the top
mapping_df = mapping_df.sort_values(
    by=['Category', 'Subcategory', 'Count'], 
    ascending=[True, True, False]
)

# Calculate the total times a Category/Subcategory pair appears
mapping_df['Total_Category_Occurrences'] = mapping_df.groupby(['Category', 'Subcategory'])['Count'].transform('sum')

# Calculate the percentage of times THIS specific mapping was used
mapping_df['Mapping_Confidence_%'] = (mapping_df['Count'] / mapping_df['Total_Category_Occurrences']) * 100

print("\nAll Mappings (Sorted by frequency):")
display(mapping_df) # Use print(mapping_df) if not in Jupyter

In [ ]:
# --- ISOLATING THE ERRORS ---
# Assuming that whatever mapping happens the majority of the time (>50%) is the "Correct" rule...
# Anything else is likely a human annotation error.

print("\n--- LIKELY HUMAN ANNOTATION ERRORS ---")
# Filter for mappings that happen less than 20% of the time for that specific category
anomalies = mapping_df[mapping_df['Mapping_Confidence_%'] < 20.0]

if anomalies.empty:
    print("Good news! No major anomalies found. The mapping is consistent.")
else:
    print("Found potential mislabels! Annotators likely messed these up:")
    display(anomalies)